&nbsp;

&nbsp;

# (Experimental) [fanta.bio](https://fanta.bio/) Expression Specificity Calculation

- Tags Per Million (TPM) データ を元に、Expression Specificity の計算を行ってみる

&nbsp;

## 参考

1. [NET-CAGE characterizes the dynamics and topology of human transcribed cis-regulatory elements](https://doi.org/10.1038/s41588-019-0485-9)
2. [An atlas of human long non-coding RNAs with accurate 5′ ends](https://doi.org/10.1038/nature21374)
3. [Recounting the FANTOM CAGE-Associated Transcriptome](https://doi.org/10.1101/gr.254656.119)
    - Code: [04expressionSpecificity.R](https://github.com/eddieimada/fcr2/blob/master/GTEX/04expressionSpecificity.R)

In [1]:
import os
import datetime
import csv
import gzip
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import plotly.express as px
import matplotlib.pyplot as plt
from IPython.display import display

pl.show_versions()
print(pl.thread_pool_size())
os.environ["POLARS_MAX_THREADS"] = "16"

--------Version info---------
Polars:              1.40.1
Index type:          UInt32
Platform:            Linux-7.0.14-orbstack-00380-ga7e0a2dc9535-aarch64-with-glibc2.36
Python:              3.13.15 (main, Aug 25 2026, 03:44:12) [GCC 12.2.0]
Runtime:             rt32

----Optional dependencies----
Azure CLI            <not installed>
adbc_driver_manager  <not installed>
altair               5.5.0
azure.identity       <not installed>
boto3                <not installed>
cloudpickle          3.1.2
connectorx           <not installed>
deltalake            <not installed>
fastexcel            <not installed>
fsspec               2026.7.0
gevent               <not installed>
google.auth          <not installed>
great_tables         <not installed>
matplotlib           3.8.4
numpy                2.3.2
openpyxl             3.1.5
pandas               2.3.2
polars_cloud         <not installed>
pyarrow              21.0.0
pydantic             2.13.5
pyiceberg            <not installed>
sqlalch

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261016


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `fanta.bio` `BED`file

In [3]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
)

display(fantabio_bed)

cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",10609,10687,"""FCHS_1""",0,""".""",10619,10687,"""0,0,255""","""cp1@DDX11L1""","""directionality:-1|class:PLA""",-1.0,"""PLA"""
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chr1""",181113,181501,"""FCHS_447416""",0,""".""",181165,181342,"""255,202,74""","""cp1@DDX11L17""","""directionality:-0.684211|class…",-0.684211,"""ELA"""
"""chr1""",190831,191016,"""FCHS_447417""",0,""".""",190900,190947,"""255,202,74""","""FCHS_447417""","""directionality:0.485714|class:…",0.485714,"""ELA"""
"""chr1""",588413,588453,"""FCHS_3""",0,""".""",588423,588453,"""255,202,74""","""cp2@LOC101928626""","""directionality:-1|class:ELA""",-1.0,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",56846801,56846977,"""FCHS_447311""",0,""".""",56846801,56846833,"""255,202,74""","""FCHS_447311""","""directionality:1|class:ELA""",1.0,"""ELA"""
"""chrY""",56855787,56855988,"""FCHS_447312""",0,""".""",56855844,56855977,"""255,202,74""","""FCHS_447312""","""directionality:0|class:ELA""",0.0,"""ELA"""
"""chrY""",56863206,56863354,"""FCHS_447313""",0,""".""",56863303,56863354,"""255,202,74""","""FCHS_447313""","""directionality:-1|class:ELA""",-1.0,"""ELA"""


In [4]:
# Map exact RGB strings to human-readable names
rgb_to_name = {
    "153,0,0": "dark_red",
    "101,0,0": "dark_red",
    "50,0,0": "dark_red",
    "204,0,0": "red",
    "255,0,0": "red",
    "0,0,51": "dark_blue",
    "0,0,102": "dark_blue",
    "0,0,153": "blue",
    "0,0,204": "blue",
    "0,0,255": "blue",
    "0,0,0": "black",
    "255,202,74": "yellow",
}

fantabio_bed_select = fantabio_bed.with_columns(
    pl.col("itemRgb").replace_strict(
        rgb_to_name,
        default="other"
    ).alias("itemRgb_name")
).select(
    "cre_id",
    "cre_name",
    "directionality",
    "fantabio_cre_class",
    "itemRgb",
    "itemRgb_name"
)

display(fantabio_bed_select)

cre_id,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,str,f64,str,str,str
"""FCHS_1""","""cp1@DDX11L1""",-1.0,"""PLA""","""0,0,255""","""blue"""
"""FCHS_2""","""cp1@WASH7P""",1.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447416""","""cp1@DDX11L17""",-0.684211,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447417""","""FCHS_447417""",0.485714,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_3""","""cp2@LOC101928626""",-1.0,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…
"""FCHS_447311""","""FCHS_447311""",1.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447312""","""FCHS_447312""",0.0,"""ELA""","""255,202,74""","""yellow"""
"""FCHS_447313""","""FCHS_447313""",-1.0,"""ELA""","""255,202,74""","""yellow"""


&nbsp;

&nbsp;

## `fanta.bio` resource (sample annotation data)

- 使用されたCAGEデータのサンプル一覧を読み込む

In [5]:
fantabio_sample_metadata = pl.read_csv(
    "../../Data/Database/fantabio/fanta.bio-CREv1.2.1.metadata.human.sdrf.txt.gz",
    separator="\t"
)

display(fantabio_sample_metadata)

Source Name,Comment [biosample_id],Characteristics [organism],Characteristics [sample type],Characteristics [description],Characteristics [CL term],Term Source REF,Term Accession Number,Characteristics [cell line],Term Source REF_duplicated_0,Term Accession Number_duplicated_0,Characteristics [EFO term],Term Source REF_duplicated_1,Term Accession Number_duplicated_1,Characteristics [tissue],Term Source REF_duplicated_2,Term Accession Number_duplicated_2,Protocol Name,Comment[CTSS count forward file],Comment[CTSS count reverse file],Comment[CTSS TPM forward file],Comment[CTSS TPM reverse file]
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CNhs10608""","""SAMD00005752""","""Homo sapiens""","""tissue sample""","""Clontech Human Universal Refer…",null,null,null,null,null,null,null,null,null,null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10608.ctss.fwd.b…","""hg38/ctss/CNhs10608.ctss.rev.b…","""hg38/ctssTpm/CNhs10608.ctssTpm…","""hg38/ctssTpm/CNhs10608.ctssTpm…"
"""CNhs10610""","""SAMD00005009""","""Homo sapiens""","""tissue sample""","""SABiosciences XpressRef Human …",null,null,null,null,null,null,null,null,null,null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10610.ctss.fwd.b…","""hg38/ctss/CNhs10610.ctss.rev.b…","""hg38/ctssTpm/CNhs10610.ctssTpm…","""hg38/ctssTpm/CNhs10610.ctssTpm…"
"""CNhs10612""","""SAMD00005100""","""Homo sapiens""","""tissue sample""","""Universal RNA - Human Normal T…",null,null,null,null,null,null,null,null,null,null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10612.ctss.fwd.b…","""hg38/ctss/CNhs10612.ctss.rev.b…","""hg38/ctssTpm/CNhs10612.ctssTpm…","""hg38/ctssTpm/CNhs10612.ctssTpm…"
"""CNhs10615""","""SAMD00005635""","""Homo sapiens""","""tissue sample""","""adipose tissue, adult, pool1""",null,null,null,null,null,null,null,null,null,"""adipose tissue""","""UBERON""","""UBERON_0001013""","""HeliscopeCAGE""","""hg38/ctss/CNhs10615.ctss.fwd.b…","""hg38/ctss/CNhs10615.ctss.rev.b…","""hg38/ctssTpm/CNhs10615.ctssTpm…","""hg38/ctssTpm/CNhs10615.ctssTpm…"
"""CNhs10616""","""SAMD00005257""","""Homo sapiens""","""tissue sample""","""bladder, adult, pool1""",null,null,null,null,null,null,null,null,null,"""urinary bladder""","""UBERON""","""UBERON_0001255""","""HeliscopeCAGE""","""hg38/ctss/CNhs10616.ctss.fwd.b…","""hg38/ctss/CNhs10616.ctss.rev.b…","""hg38/ctssTpm/CNhs10616.ctssTpm…","""hg38/ctssTpm/CNhs10616.ctssTpm…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX28551190""","""SAMN48137815""","""Homo sapiens""","""cell culture""","""NIPBL knockdown 501mel cells (…",null,null,null,"""501-mel""","""Cellosaurus""","""CVCL_4633""",null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX28551190.ctss.fwd…","""hg38/ctss/SRX28551190.ctss.rev…","""hg38/ctssTpm/SRX28551190.ctssT…","""hg38/ctssTpm/SRX28551190.ctssT…"
"""SRX28551189""","""SAMN48137816""","""Homo sapiens""","""cell culture""","""NIPBL knockdown 501mel cells (…",null,null,null,"""501-mel""","""Cellosaurus""","""CVCL_4633""",null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX28551189.ctss.fwd…","""hg38/ctss/SRX28551189.ctss.rev…","""hg38/ctssTpm/SRX28551189.ctssT…","""hg38/ctssTpm/SRX28551189.ctssT…"
"""SRX28551188""","""SAMN48137817""","""Homo sapiens""","""cell culture""","""NIPBL knockdown 501mel cells (…",null,null,null,"""501-mel""","""Cellosaurus""","""CVCL_4633""",null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX28551188.ctss.fwd…","""hg38/ctss/SRX28551188.ctss.rev…","""hg38/ctssTpm/SRX28551188.ctssT…","""hg38/ctssTpm/SRX28551188.ctssT…"


&nbsp;

&nbsp;

## Sample Classification

- cell 由来のサンプルと、tissue 由来のサンプルに分ける

In [6]:
fantabio_sample_cell = fantabio_sample_metadata.filter(
    (pl.col("Characteristics [CL term]").is_not_null()) |
    (pl.col("Characteristics [cell line]").is_not_null())
).sort(
    by=["Term Accession Number"],
    descending=True
)

display(fantabio_sample_cell)

Source Name,Comment [biosample_id],Characteristics [organism],Characteristics [sample type],Characteristics [description],Characteristics [CL term],Term Source REF,Term Accession Number,Characteristics [cell line],Term Source REF_duplicated_0,Term Accession Number_duplicated_0,Characteristics [EFO term],Term Source REF_duplicated_1,Term Accession Number_duplicated_1,Characteristics [tissue],Term Source REF_duplicated_2,Term Accession Number_duplicated_2,Protocol Name,Comment[CTSS count forward file],Comment[CTSS count reverse file],Comment[CTSS TPM forward file],Comment[CTSS TPM reverse file]
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CNhs10722""","""SAMD00005753""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10722.ctss.fwd.b…","""hg38/ctss/CNhs10722.ctss.rev.b…","""hg38/ctssTpm/CNhs10722.ctssTpm…","""hg38/ctssTpm/CNhs10722.ctssTpm…"
"""CNhs10723""","""SAMD00005601""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10723.ctss.fwd.b…","""hg38/ctss/CNhs10723.ctss.rev.b…","""hg38/ctssTpm/CNhs10723.ctssTpm…","""hg38/ctssTpm/CNhs10723.ctssTpm…"
"""CNhs10724""","""SAMD00004824""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10724.ctss.fwd.b…","""hg38/ctss/CNhs10724.ctss.rev.b…","""hg38/ctssTpm/CNhs10724.ctssTpm…","""hg38/ctssTpm/CNhs10724.ctssTpm…"
"""CNhs10726""","""SAMD00005224""","""Homo sapiens""","""cell culture""","""lung adenocarcinoma cell line:…",null,null,null,"""PC-14""","""Cellosaurus""","""CVCL_1640""","""PC-14""","""EFO""","""EFO_0002844""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10726.ctss.fwd.b…","""hg38/ctss/CNhs10726.ctss.rev.b…","""hg38/ctssTpm/CNhs10726.ctssTpm…","""hg38/ctssTpm/CNhs10726.ctssTpm…"
"""CNhs10727""","""SAMD00005883""","""Homo sapiens""","""cell culture""","""chronic myelogenous leukemia c…",null,null,null,"""KU812""","""Cellosaurus""","""CVCL_0379""","""KU812""","""EFO""","""EFO_0002222""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10727.ctss.fwd.b…","""hg38/ctss/CNhs10727.ctss.rev.b…","""hg38/ctssTpm/CNhs10727.ctssTpm…","""hg38/ctssTpm/CNhs10727.ctssTpm…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CNhs13227""","""SAMD00005368""","""Homo sapiens""","""primary cell""","""CD34+ Progenitors, donor1""","""stem cell""","""CL""","""CL_0000034""",null,null,null,null,null,null,null,null,null,"""LQ-HeliscopeCAGE""","""hg38/ctss/CNhs13227.ctss.fwd.b…","""hg38/ctss/CNhs13227.ctss.rev.b…","""hg38/ctssTpm/CNhs13227.ctssTpm…","""hg38/ctssTpm/CNhs13227.ctssTpm…"
"""CNhs12205""","""SAMD00005630""","""Homo sapiens""","""primary cell""","""CD34+ Progenitors, donor2""","""stem cell""","""CL""","""CL_0000034""",null,null,null,null,null,null,null,null,null,"""LQ-HeliscopeCAGE""","""hg38/ctss/CNhs12205.ctss.fwd.b…","""hg38/ctss/CNhs12205.ctss.rev.b…","""hg38/ctssTpm/CNhs12205.ctssTpm…","""hg38/ctssTpm/CNhs12205.ctssTpm…"
"""CNhs12552""","""SAMD00005606""","""Homo sapiens""","""primary cell""","""CD133+ stem cells - adult bone…","""stem cell""","""CL""","""CL_0000034""",null,null,null,null,null,null,null,null,null,"""LQ-HeliscopeCAGE""","""hg38/ctss/CNhs12552.ctss.fwd.b…","""hg38/ctss/CNhs12552.ctss.rev.b…","""hg38/ctssTpm/CNhs12552.ctssTpm…","""hg38/ctssTpm/CNhs12552.ctssTpm…"


In [7]:
fantabio_sample_uberon = fantabio_sample_metadata.filter(
    pl.col("Characteristics [tissue]").is_not_null()
).sort(
    by=["Term Accession Number"],
    descending=True
)

display(fantabio_sample_uberon)

Source Name,Comment [biosample_id],Characteristics [organism],Characteristics [sample type],Characteristics [description],Characteristics [CL term],Term Source REF,Term Accession Number,Characteristics [cell line],Term Source REF_duplicated_0,Term Accession Number_duplicated_0,Characteristics [EFO term],Term Source REF_duplicated_1,Term Accession Number_duplicated_1,Characteristics [tissue],Term Source REF_duplicated_2,Term Accession Number_duplicated_2,Protocol Name,Comment[CTSS count forward file],Comment[CTSS count reverse file],Comment[CTSS TPM forward file],Comment[CTSS TPM reverse file]
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX23366460""","""SAMN39578383""","""Homo sapiens""","""tissue sample""","""facial mesenchyme from 10-week…",null,null,null,null,null,null,null,null,null,"""facial mesenchyme""","""UBERON""","""UBERON_0009891""","""CAGE""","""hg38/ctss/SRX23366460.ctss.fwd…","""hg38/ctss/SRX23366460.ctss.rev…","""hg38/ctssTpm/SRX23366460.ctssT…","""hg38/ctssTpm/SRX23366460.ctssT…"
"""SRX23366461""","""SAMN39578382""","""Homo sapiens""","""tissue sample""","""facial mesenchyme from 5–6-wee…",null,null,null,null,null,null,null,null,null,"""facial mesenchyme""","""UBERON""","""UBERON_0009891""","""CAGE""","""hg38/ctss/SRX23366461.ctss.fwd…","""hg38/ctss/SRX23366461.ctss.rev…","""hg38/ctssTpm/SRX23366461.ctssT…","""hg38/ctssTpm/SRX23366461.ctssT…"
"""SRX23366462""","""SAMN39578381""","""Homo sapiens""","""tissue sample""","""facial mesenchyme from 5–6-wee…",null,null,null,null,null,null,null,null,null,"""facial mesenchyme""","""UBERON""","""UBERON_0009891""","""CAGE""","""hg38/ctss/SRX23366462.ctss.fwd…","""hg38/ctss/SRX23366462.ctss.rev…","""hg38/ctssTpm/SRX23366462.ctssT…","""hg38/ctssTpm/SRX23366462.ctssT…"
"""SRX23366463""","""SAMN39578380""","""Homo sapiens""","""tissue sample""","""facial mesenchyme from 10-week…",null,null,null,null,null,null,null,null,null,"""facial mesenchyme""","""UBERON""","""UBERON_0009891""","""CAGE""","""hg38/ctss/SRX23366463.ctss.fwd…","""hg38/ctss/SRX23366463.ctss.rev…","""hg38/ctssTpm/SRX23366463.ctssT…","""hg38/ctssTpm/SRX23366463.ctssT…"
"""SRX23366464""","""SAMN39578379""","""Homo sapiens""","""tissue sample""","""facial mesenchyme from 5–6-wee…",null,null,null,null,null,null,null,null,null,"""facial mesenchyme""","""UBERON""","""UBERON_0009891""","""CAGE""","""hg38/ctss/SRX23366464.ctss.fwd…","""hg38/ctss/SRX23366464.ctss.rev…","""hg38/ctssTpm/SRX23366464.ctssT…","""hg38/ctssTpm/SRX23366464.ctssT…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CNhs10619""","""SAMD00005815""","""Homo sapiens""","""tissue sample""","""colon, adult, pool1""",null,null,null,null,null,null,null,null,null,"""colon""","""UBERON""","""UBERON_0001155""","""HeliscopeCAGE""","""hg38/ctss/CNhs10619.ctss.fwd.b…","""hg38/ctss/CNhs10619.ctss.rev.b…","""hg38/ctssTpm/CNhs10619.ctssTpm…","""hg38/ctssTpm/CNhs10619.ctssTpm…"
"""CNhs10618""","""SAMD00004902""","""Homo sapiens""","""tissue sample""","""cervix, adult, pool1""",null,null,null,null,null,null,null,null,null,"""uterine cervix""","""UBERON""","""UBERON_0000002""","""HeliscopeCAGE""","""hg38/ctss/CNhs10618.ctss.fwd.b…","""hg38/ctss/CNhs10618.ctss.rev.b…","""hg38/ctssTpm/CNhs10618.ctssTpm…","""hg38/ctssTpm/CNhs10618.ctssTpm…"
"""CNhs10617""","""SAMD00005144""","""Homo sapiens""","""tissue sample""","""brain, adult, pool1""",null,null,null,null,null,null,null,null,null,"""brain""","""UBERON""","""UBERON_0000955""","""HeliscopeCAGE""","""hg38/ctss/CNhs10617.ctss.fwd.b…","""hg38/ctss/CNhs10617.ctss.rev.b…","""hg38/ctssTpm/CNhs10617.ctssTpm…","""hg38/ctssTpm/CNhs10617.ctssTpm…"


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `fanta.bio` resource (Tags per million (TPM) data)

- 熱ストレス下のCAGEデータと比較できるかもしれないので、読み込んでおく
- 栂さんから教えてもらった[NET-CAGEの論文](https://doi.org/10.1038/s41588-019-0485-9)のFig.6に記載がある

In [8]:
tpm_path = "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.tpm.txt.gz"
# Skip ## comment lines. read_csv n_rows counts those lines, so n_rows=1 returns no data.
with gzip.open(tpm_path, "rt") as handle:
    tpm_columns = next(
        csv.reader(
            (line for line in handle if not line.startswith("##")),
            delimiter="\t",
        )
    )

# Sample columns are TPM. Lock them to Float64 so integer-looking rows are not inferred as Int64.
fantabio_tpm = pl.read_csv(
    tpm_path,
    separator="\t",
    n_threads=16,
    comment_prefix="##",
    schema={
        name: pl.String if name == "creRegion" else pl.Float64
        for name in tpm_columns
    },
).with_columns(
    # "creRegion" カラムの内容を分割して、最初の要素 (fanta.bio の CRE ID) を "cre_id" カラムとして抽出
    pl.col(
        "creRegion"
    ).str.split(
        "|"
    ).list.first().alias(
        "creRegion"
    )
).rename(
    {
        "creRegion": "cre_id"
    }
)

display(fantabio_tpm)

cre_id,DRX199485,DRX199486,DRX199487,DRX199488,DRX199489,DRX199490,DRX199491,DRX199492,DRX199493,DRX199494,DRX199495,DRX199496,DRX199497,DRX199498,DRX199499,DRX199500,DRX199501,DRX199502,DRX199503,DRX199504,DRX199505,DRX199506,DRX199507,DRX199508,DRX319592,DRX319593,DRX319594,DRX319595,DRX319596,DRX319597,DRX319598,DRX319599,DRX319600,DRX319601,DRX319602,DRX319603,…,CNhs14586,CNhs14587,CNhs14588,CNhs14589,CNhs14590,CNhs14591,CNhs14592,CNhs14594,CNhs14595,CNhs14596,CNhs14597,CNhs14598,CNhs14599,CNhs14600,CNhs14601,CNhs14602,CNhs14603,CNhs14604,CNhs14605,CNhs14606,CNhs14607,CNhs14609,CNhs14610,CNhs14611,CNhs14612,CNhs14613,CNhs14618,CNhi10525_AGT,CNhi10786_TAC,CNhi10789_ATG,CNhi10939_ACC,CNhi10939_ACG,CNhi10941_CAC,CNhi10941_GCT,LShi10008_ACAGTG-CACTAGT,LShi10030_ACAGTG-CACTATG,LShi10030_ACTTGA-CACTAGT
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""FCHS_1""",0.0,0.0,0.0,0.0,0.0,0.0,0.288724,0.291955,0.0,0.302918,0.0,0.964758,0.742881,0.355559,1.364579,0.14098,0.0,0.0,0.0,1.681064,0.662089,0.717855,1.329518,1.229479,0.0,0.0,0.694102,0.0,0.0,0.0,0.791634,0.0,0.0,0.0,0.911456,2.950016,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.096454,0.133485,0.142706,0.0,0.0,0.0
"""FCHS_2""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.289308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.529031
"""FCHS_447416""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.124354,0.0,0.0,0.0,0.113715,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.530689,0.0,0.0,0.0,0.0,0.0,0.34794,0.0,0.0,0.0,0.0,0.185338,0.0,0.0,0.444188,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""FCHS_447417""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.557673,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.379557,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""FCHS_3""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_447311""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.37782,0.0,0.0,0.0,0.0,0.0,0.530689,0.242888,0.257778,0.0,0.0,0.0,0.0,0.461449,0.0,0.0,0.0,0.0,0.0,0.342374,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""FCHS_447312""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.755639,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.34794,0.0,0.0,0.0,0.0,0.185338,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""FCHS_447313""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.428118,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.370676,0.0,0.0,0.0,0.344186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Test (1)

- 例えば、THP-1 cell について考える
- まず、THP-1 に該当する Source name をリストとして取得する
- その後、そのリストに該当するカラムを、`fanta.bio_tpm` データフレームから探す

In [9]:
THP1_cell = fantabio_sample_cell.filter(
    pl.col("Characteristics [EFO term]") == "THP-1"
)

THP1_cell_source_names = THP1_cell["Source Name"].to_list()

display(THP1_cell)

Source Name,Comment [biosample_id],Characteristics [organism],Characteristics [sample type],Characteristics [description],Characteristics [CL term],Term Source REF,Term Accession Number,Characteristics [cell line],Term Source REF_duplicated_0,Term Accession Number_duplicated_0,Characteristics [EFO term],Term Source REF_duplicated_1,Term Accession Number_duplicated_1,Characteristics [tissue],Term Source REF_duplicated_2,Term Accession Number_duplicated_2,Protocol Name,Comment[CTSS count forward file],Comment[CTSS count reverse file],Comment[CTSS TPM forward file],Comment[CTSS TPM reverse file]
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CNhs10722""","""SAMD00005753""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10722.ctss.fwd.b…","""hg38/ctss/CNhs10722.ctss.rev.b…","""hg38/ctssTpm/CNhs10722.ctssTpm…","""hg38/ctssTpm/CNhs10722.ctssTpm…"
"""CNhs10723""","""SAMD00005601""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10723.ctss.fwd.b…","""hg38/ctss/CNhs10723.ctss.rev.b…","""hg38/ctssTpm/CNhs10723.ctssTpm…","""hg38/ctssTpm/CNhs10723.ctssTpm…"
"""CNhs10724""","""SAMD00004824""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10724.ctss.fwd.b…","""hg38/ctss/CNhs10724.ctss.rev.b…","""hg38/ctssTpm/CNhs10724.ctssTpm…","""hg38/ctssTpm/CNhs10724.ctssTpm…"
"""CNhs14556""","""SAMD00022299""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs14556.ctss.fwd.b…","""hg38/ctss/CNhs14556.ctss.rev.b…","""hg38/ctssTpm/CNhs14556.ctssTpm…","""hg38/ctssTpm/CNhs14556.ctssTpm…"
"""SRX6586383""","""SRS5152078""","""Homo sapiens""","""cell culture""","""CNhi10326_THP-1_Non-treatment …",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""nAnT-iCAGE""","""hg38/ctss/SRX6586383.ctss.fwd.…","""hg38/ctss/SRX6586383.ctss.rev.…","""hg38/ctssTpm/SRX6586383.ctssTp…","""hg38/ctssTpm/SRX6586383.ctssTp…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX6586232""","""SRS5151927""","""Homo sapiens""","""cell culture""","""CNhi10327_THP-1_Fluvastatin-30…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""nAnT-iCAGE""","""hg38/ctss/SRX6586232.ctss.fwd.…","""hg38/ctss/SRX6586232.ctss.rev.…","""hg38/ctssTpm/SRX6586232.ctssTp…","""hg38/ctssTpm/SRX6586232.ctssTp…"
"""SRX6586231""","""SRS5151926""","""Homo sapiens""","""cell culture""","""CNhi10326_THP-1_Simvastatin-30…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""nAnT-iCAGE""","""hg38/ctss/SRX6586231.ctss.fwd.…","""hg38/ctss/SRX6586231.ctss.rev.…","""hg38/ctssTpm/SRX6586231.ctssTp…","""hg38/ctssTpm/SRX6586231.ctssTp…"
"""SRX6586230""","""SRS5151925""","""Homo sapiens""","""cell culture""","""CNhi10326_THP-1_Simvastatin-30…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""nAnT-iCAGE""","""hg38/ctss/SRX6586230.ctss.fwd.…","""hg38/ctss/SRX6586230.ctss.rev.…","""hg38/ctssTpm/SRX6586230.ctssTp…","""hg38/ctssTpm/SRX6586230.ctssTp…"


&nbsp;

&nbsp;

## Test (2)

- 平均のTPM値をサンプルごとに計算する (今回は THP-1)
- このデータフレーム、cre_id カラムと サンプルごとに計算した平均のTPM値 のカラムを持つデータフレームを作成し、それを cre_id をキーに `join` していくイメージ
- 最終的に、サンプルごとの平均TPM値を持つデータフレームを作成する
- この後に、エントロピーの計算、および、expression specificity を計算するイメージ


In [10]:
fantabio_tpm_thp1 = fantabio_tpm.select(
   "cre_id",
   # Same select expression as fantabio_tpm_mean. with_columns changes the float bits.
   pl.mean_horizontal(sorted(THP1_cell_source_names)).alias(
      "mean_tpm_" + THP1_cell.select("Term Accession Number_duplicated_0").unique().item()
   ),
)

display(fantabio_tpm_thp1)

cre_id,mean_tpm_CVCL_0006
str,f64
"""FCHS_1""",0.847295
"""FCHS_2""",0.0
"""FCHS_447416""",0.007199
"""FCHS_447417""",0.0
"""FCHS_3""",0.0
…,…
"""FCHS_447311""",0.0
"""FCHS_447312""",0.0
"""FCHS_447313""",0.0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 全てのサンプルへの拡張

### 1. サンプルと注釈を、見やすい列名の表にする

- THP-1 の元の列順を保つため、既存の cell 表 → tissue 表 → 残りの metadata の順で並べる。
- 重複して登場した同じ `Source Name` は一度だけ使う。元の metadata 自体に重複 ID があれば停止する。

In [11]:
assert fantabio_sample_metadata["Source Name"].null_count() == 0
assert fantabio_sample_metadata["Source Name"].n_unique() == fantabio_sample_metadata.height


fantabio_sample_ordered = (
    pl
    .concat(
        [
            fantabio_sample_cell, fantabio_sample_uberon, fantabio_sample_metadata,
        ]
    )
    .unique(
        subset="Source Name",
        keep="first",
        maintain_order=True
    )
)

display(fantabio_sample_ordered)

Source Name,Comment [biosample_id],Characteristics [organism],Characteristics [sample type],Characteristics [description],Characteristics [CL term],Term Source REF,Term Accession Number,Characteristics [cell line],Term Source REF_duplicated_0,Term Accession Number_duplicated_0,Characteristics [EFO term],Term Source REF_duplicated_1,Term Accession Number_duplicated_1,Characteristics [tissue],Term Source REF_duplicated_2,Term Accession Number_duplicated_2,Protocol Name,Comment[CTSS count forward file],Comment[CTSS count reverse file],Comment[CTSS TPM forward file],Comment[CTSS TPM reverse file]
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CNhs10722""","""SAMD00005753""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10722.ctss.fwd.b…","""hg38/ctss/CNhs10722.ctss.rev.b…","""hg38/ctssTpm/CNhs10722.ctssTpm…","""hg38/ctssTpm/CNhs10722.ctssTpm…"
"""CNhs10723""","""SAMD00005601""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10723.ctss.fwd.b…","""hg38/ctss/CNhs10723.ctss.rev.b…","""hg38/ctssTpm/CNhs10723.ctssTpm…","""hg38/ctssTpm/CNhs10723.ctssTpm…"
"""CNhs10724""","""SAMD00004824""","""Homo sapiens""","""cell culture""","""acute myeloid leukemia (FAB M5…",null,null,null,"""THP-1""","""Cellosaurus""","""CVCL_0006""","""THP-1""","""EFO""","""EFO_0001253""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10724.ctss.fwd.b…","""hg38/ctss/CNhs10724.ctss.rev.b…","""hg38/ctssTpm/CNhs10724.ctssTpm…","""hg38/ctssTpm/CNhs10724.ctssTpm…"
"""CNhs10726""","""SAMD00005224""","""Homo sapiens""","""cell culture""","""lung adenocarcinoma cell line:…",null,null,null,"""PC-14""","""Cellosaurus""","""CVCL_1640""","""PC-14""","""EFO""","""EFO_0002844""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10726.ctss.fwd.b…","""hg38/ctss/CNhs10726.ctss.rev.b…","""hg38/ctssTpm/CNhs10726.ctssTpm…","""hg38/ctssTpm/CNhs10726.ctssTpm…"
"""CNhs10727""","""SAMD00005883""","""Homo sapiens""","""cell culture""","""chronic myelogenous leukemia c…",null,null,null,"""KU812""","""Cellosaurus""","""CVCL_0379""","""KU812""","""EFO""","""EFO_0002222""",null,null,null,"""HeliscopeCAGE""","""hg38/ctss/CNhs10727.ctss.fwd.b…","""hg38/ctss/CNhs10727.ctss.rev.b…","""hg38/ctssTpm/CNhs10727.ctssTpm…","""hg38/ctssTpm/CNhs10727.ctssTpm…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX11179953""","""SAMN19775304""","""Homo sapiens""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX11179953.ctss.fwd…","""hg38/ctss/SRX11179953.ctss.rev…","""hg38/ctssTpm/SRX11179953.ctssT…","""hg38/ctssTpm/SRX11179953.ctssT…"
"""SRX11179954""","""SAMN19775303""","""Homo sapiens""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX11179954.ctss.fwd…","""hg38/ctss/SRX11179954.ctss.rev…","""hg38/ctssTpm/SRX11179954.ctssT…","""hg38/ctssTpm/SRX11179954.ctssT…"
"""SRX11179955""","""SAMN19775302""","""Homo sapiens""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,null,"""CAGE""","""hg38/ctss/SRX11179955.ctss.fwd…","""hg38/ctss/SRX11179955.ctss.rev…","""hg38/ctssTpm/SRX11179955.ctssT…","""hg38/ctssTpm/SRX11179955.ctssT…"


In [12]:
fantabio_sample_annotation = (
    fantabio_sample_ordered
    .select(
        "Source Name",
        "Comment [biosample_id]",
        "Protocol Name",
        "Characteristics [sample type]",
        "Characteristics [description]",
        pl.col("Term Accession Number_duplicated_0").alias("cell_line_id"),
        pl.col("Term Accession Number").alias("cell_id"),
        pl.col("Term Accession Number_duplicated_2").alias("tissue_id"),
        pl.col("Term Accession Number_duplicated_1").alias("efo_id"),
        pl.col("Characteristics [cell line]").alias("cell_line_name"),
        pl.col("Characteristics [CL term]").alias("cell_name"),
        pl.col("Characteristics [tissue]").alias("tissue_name"),
        pl.col("Characteristics [EFO term]").alias("efo_name"),
    )
    .with_columns(
        pl.col("cell_line_id", "cell_id", "tissue_id", "efo_id")
        .str.strip_chars().replace("", None)
    )
)

display(fantabio_sample_annotation.head(5))

Source Name,Comment [biosample_id],Protocol Name,Characteristics [sample type],Characteristics [description],cell_line_id,cell_id,tissue_id,efo_id,cell_line_name,cell_name,tissue_name,efo_name
str,str,str,str,str,str,str,str,str,str,str,str,str
"""CNhs10722""","""SAMD00005753""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1"""
"""CNhs10723""","""SAMD00005601""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1"""
"""CNhs10724""","""SAMD00004824""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1"""
"""CNhs10726""","""SAMD00005224""","""HeliscopeCAGE""","""cell culture""","""lung adenocarcinoma cell line:…","""CVCL_1640""",null,null,"""EFO_0002844""","""PC-14""",null,null,"""PC-14"""
"""CNhs10727""","""SAMD00005883""","""HeliscopeCAGE""","""cell culture""","""chronic myelogenous leukemia c…","""CVCL_0379""",null,null,"""EFO_0002222""","""KU812""",null,null,"""KU812"""


In [13]:
SRX11179953 = fantabio_sample_annotation.filter(
    pl.col("Source Name") == "SRX11179953"
)

display(SRX11179953)

Source Name,Comment [biosample_id],Protocol Name,Characteristics [sample type],Characteristics [description],cell_line_id,cell_id,tissue_id,efo_id,cell_line_name,cell_name,tissue_name,efo_name
str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX11179953""","""SAMN19775304""","""CAGE""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null


&nbsp;

&nbsp;

&nbsp;

### 2. sample を一つの集約群に割り当てる

- ID の優先順位は **Cellosaurus → CL → UBERON → EFO**。
- 例えば細胞株と由来組織の両方が注釈されていても、細胞株の群に一度だけ入れる。

In [14]:
name_only_cell_line = (
    pl.all_horizontal(pl.col("cell_line_id", "cell_id", "tissue_id", "efo_id").is_null())
    & pl.col("cell_line_name").is_not_null()
)

fantabio_sample_assignment = (
    fantabio_sample_annotation
    .with_columns(pl.coalesce(
        "cell_line_id", "cell_id", "tissue_id", "efo_id"
    ).alias("group_id"))

    .with_columns(
        pl
        .when(
            pl.col("cell_line_id").is_not_null()
        ).then(pl.lit("cell_line"))

        .when(
            pl.col("cell_id").is_not_null()
        ).then(pl.lit("cell"))

        .when(
            pl.col("tissue_id").is_not_null()
        ).then(pl.lit("tissue"))

        .when(
            pl.col("efo_id").is_not_null()
        ).then(pl.lit("EFO_only"))

        .when(
            name_only_cell_line
        ).then(pl.lit("cell_line"))

        .otherwise(None).alias("group_type"),
        
        pl
        .when(
            pl.col("cell_line_id").is_not_null()
        ).then(pl.col("cell_line_name"))
        
        .when(
            pl.col("cell_id").is_not_null()
        ).then(pl.col("cell_name"))
        
        .when(
            pl.col("tissue_id").is_not_null()
        ).then(pl.col("tissue_name"))

        .when(
            name_only_cell_line
        ).then(pl.col("cell_line_name"))
    
        .otherwise(
            pl.col("efo_name")
        ).fill_null(pl.col("group_id")).alias("group_name"),
        
        pl.col("Source Name").is_in(fantabio_tpm.columns).alias("in_tpm"),
    )


)

display(fantabio_sample_assignment)

Source Name,Comment [biosample_id],Protocol Name,Characteristics [sample type],Characteristics [description],cell_line_id,cell_id,tissue_id,efo_id,cell_line_name,cell_name,tissue_name,efo_name,group_id,group_type,group_name,in_tpm
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool
"""CNhs10722""","""SAMD00005753""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1""","""CVCL_0006""","""cell_line""","""THP-1""",true
"""CNhs10723""","""SAMD00005601""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1""","""CVCL_0006""","""cell_line""","""THP-1""",true
"""CNhs10724""","""SAMD00004824""","""HeliscopeCAGE""","""cell culture""","""acute myeloid leukemia (FAB M5…","""CVCL_0006""",null,null,"""EFO_0001253""","""THP-1""",null,null,"""THP-1""","""CVCL_0006""","""cell_line""","""THP-1""",true
"""CNhs10726""","""SAMD00005224""","""HeliscopeCAGE""","""cell culture""","""lung adenocarcinoma cell line:…","""CVCL_1640""",null,null,"""EFO_0002844""","""PC-14""",null,null,"""PC-14""","""CVCL_1640""","""cell_line""","""PC-14""",true
"""CNhs10727""","""SAMD00005883""","""HeliscopeCAGE""","""cell culture""","""chronic myelogenous leukemia c…","""CVCL_0379""",null,null,"""EFO_0002222""","""KU812""",null,null,"""KU812""","""CVCL_0379""","""cell_line""","""KU812""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX11179953""","""SAMN19775304""","""CAGE""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,true
"""SRX11179954""","""SAMN19775303""","""CAGE""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,true
"""SRX11179955""","""SAMN19775302""","""CAGE""","""cell culture""","""Microglia cell line C13NJ cell…",null,null,null,null,null,null,null,null,null,null,null,true


&nbsp;

&nbsp;

&nbsp;

### 3. 対応しないサンプルを表示する

- `group_id`カラムに値がアサインされない行を表示する

In [16]:
fantabio_unassigned_samples = fantabio_sample_assignment.filter(pl.col("group_id").is_null())


pl.Config.set_tbl_rows(250)
pl.Config.set_fmt_str_lengths(200)
print(fantabio_unassigned_samples.select("Characteristics [description]").sort("Characteristics [description]").unique())

pl.Config.set_tbl_rows(2)
display(fantabio_unassigned_samples)

shape: (291, 1)
┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ Characteristics [description]                                                                    │
│ ---                                                                                              │
│ str                                                                                              │
╞══════════════════════════════════════════════════════════════════════════════════════════════════╡
│ 293SLAM rinderpest infection, 00hr, biol_rep1                                                    │
│ 293SLAM rinderpest infection, 00hr, biol_rep2                                                    │
│ 293SLAM rinderpest infection, 00hr, biol_rep3                                                    │
│ 293SLAM rinderpest infection, 06hr, biol_rep1                                                    │
│ 293SLAM rinderpest infection, 06hr, biol_rep2                            

Source Name,Comment [biosample_id],Protocol Name,Characteristics [sample type],Characteristics [description],cell_line_id,cell_id,tissue_id,efo_id,cell_line_name,cell_name,tissue_name,efo_name,group_id,group_type,group_name,in_tpm
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool
"""CNhs11045""","""SAMD00004751""","""HeliscopeCAGE""","""cell culture""","""cord blood derived cell line:COBL-a untreated""",null,null,null,null,"""COBL-a""",null,null,null,null,"""cell_line""","""COBL-a""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX11179957""","""SAMN19775300""","""CAGE""","""cell culture""","""Microglia cell line C13NJ cells_5uM JQ1 (BET inhibitor) for 24h_R3""",null,null,null,null,null,null,null,null,null,null,null,true


&nbsp;

&nbsp;

&nbsp;

### 4. group ID → 対象サンプル column の対応表を作る

- `source_names` は Polars の List 列。THP-1 の `THP1_cell_source_names` に相当するリストを全群について保持する
- `n_samples` はライブラリー数であり、独立な donor 数ではない
- 同じ ID に複数のラベルがある場合も、`group_names` にすべて残す

In [17]:
fantabio_group_metadata = (
    fantabio_sample_assignment
    .filter(pl.col("group_id").is_not_null()) # 一旦グループに割り当てられたサンプルのみを対象にする
    
    .group_by("group_id", maintain_order=True)
    .agg(
        pl.col("group_name").unique(maintain_order=True).alias("group_names"),
        pl.col("group_type").first(),
        pl.col("Source Name").alias("source_names"),
        pl.len().alias("n_samples"),
        pl.col("Protocol Name").unique().sort().alias("protocols"),
        pl.col("Characteristics [sample type]").unique().sort().alias("sample_types"),
    )
    .with_columns(
    (pl.lit("mean_tpm_") + pl.col("group_id")).alias("mean_column"),
    pl.col("protocols").list.len().alias("n_protocols"),
    )

    .sort("n_samples", descending=True)
)


display(fantabio_group_metadata)
print("Group Count:", fantabio_group_metadata.height)

group_id,group_names,group_type,source_names,n_samples,protocols,sample_types,mean_column,n_protocols
str,list[str],str,list[str],u32,list[str],list[str],str,u32
"""CVCL_0031""","[""MCF-7""]","""cell_line""","[""CNhs11943"", ""CNhs12564"", … ""SRX19143101""]",256,"[""CAGE"", ""HeliscopeCAGE"", … ""nAnT-iCAGE""]","[""cell culture""]","""mean_tpm_CVCL_0031""",5
…,…,…,…,…,…,…,…,…
"""UBERON_0001255""","[""urinary bladder""]","""tissue""","[""CNhs10616""]",1,"[""HeliscopeCAGE""]","[""tissue sample""]","""mean_tpm_UBERON_0001255""",1


Group Count: 519


&nbsp;

&nbsp;

&nbsp;

### 5.  Check TPM Value

- ここだけは、注釈に使う全サンプル・全 CRE を検査する。巨大な値の表を long format に変換するのではなく、サンプルごとの異常値の件数だけを小さい表にす
- `null`・NaN・Inf・負値があれば停止し、ゼロへの置換や無視はしない
- 元の THP-1 平均と同じ結果を検証する対象は、この検査を通る入力である

In [18]:
used_source_names = fantabio_sample_assignment["Source Name"].to_list()

fantabio_tpm_qc = (
    fantabio_tpm.lazy().select(used_source_names)
    .select((pl.all().is_null() | ~pl.all().is_finite() | (pl.all() < 0)).sum())
    .collect()
    .unpivot(variable_name="Source Name", value_name="n_invalid")
)

display(fantabio_tpm_qc.filter(pl.col("n_invalid") > 0))
assert fantabio_tpm_qc["n_invalid"].sum() == 0, "無効な TPM が含まれています。"
assert fantabio_tpm["cre_id"].null_count() == 0
assert fantabio_tpm["cre_id"].n_unique() == fantabio_tpm.height

Source Name,n_invalid
str,u32


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 6. 全てのグループの mean TPM を持つ Dataframe を作成する

- THP-1 の `pl.mean_horizontal(source_names)` を、全群について同じように指定する
- 下の `for` は**小さい metadata 表から列の計算式を並べるだけ**で、CRE の数値を Python のループで計算しない
- 元の表の行順が共通なので、群ごとに `join` を繰り返す代わりに、一度の `select` で横に並べられる

In [19]:
fantabio_tpm_mean = fantabio_tpm.lazy().select(
    "cre_id",
    *[
        pl.mean_horizontal(sorted(source_names)).alias(mean_column)
        for mean_column, source_names in fantabio_group_metadata.select(
            "mean_column", "source_names"
        ).iter_rows()
    ],
)

display(fantabio_tpm_mean.head(5).collect())

cre_id,mean_tpm_CVCL_0031,mean_tpm_UBERON_0000948,mean_tpm_CL_0000235,mean_tpm_CVCL_0006,mean_tpm_CL_0002539,mean_tpm_CVCL_0004,mean_tpm_CVCL_0145,mean_tpm_CVCL_0027,mean_tpm_CVCL_0548,mean_tpm_CL_0002372,mean_tpm_CL_0002138,mean_tpm_UBERON_0016525,mean_tpm_UBERON_0001134,mean_tpm_CL_0001054,mean_tpm_CVCL_0023,mean_tpm_CVCL_1547,mean_tpm_CVCL_9U31,mean_tpm_EFO_0004905,mean_tpm_CVCL_7526,mean_tpm_CL_0002620,mean_tpm_CL_0000576,mean_tpm_CL_0000136,mean_tpm_UBERON_0009891,mean_tpm_CVCL_0022,mean_tpm_CL_0002600,mean_tpm_CL_0002552,mean_tpm_CL_0000134,mean_tpm_CVCL_0395,mean_tpm_CVCL_4633,mean_tpm_CL_0000625,mean_tpm_CL_0000236,mean_tpm_CVCL_1906,mean_tpm_CL_0002551,mean_tpm_CL_0000097,mean_tpm_CL_0000081,mean_tpm_CVCL_0556,…,mean_tpm_UBERON_0001638,mean_tpm_UBERON_0001637,mean_tpm_UBERON_0001154,mean_tpm_UBERON_0010743,mean_tpm_UBERON_0000995,mean_tpm_UBERON_0001044,mean_tpm_UBERON_0002369,mean_tpm_UBERON_0000310,mean_tpm_UBERON_0002079,mean_tpm_UBERON_0002084,mean_tpm_UBERON_0000029,mean_tpm_UBERON_0001103,mean_tpm_UBERON_0001052,mean_tpm_UBERON_0000945,mean_tpm_UBERON_0002331,mean_tpm_UBERON_0005795,mean_tpm_UBERON_0000970,mean_tpm_UBERON_0000178,mean_tpm_UBERON_0000947,mean_tpm_UBERON_0001264,mean_tpm_UBERON_0001135,mean_tpm_UBERON_0002372,mean_tpm_UBERON_0002336,mean_tpm_UBERON_0002363,mean_tpm_UBERON_0002022,mean_tpm_UBERON_0001882,mean_tpm_UBERON_0002902,mean_tpm_UBERON_0007190,mean_tpm_UBERON_0000988,mean_tpm_UBERON_0002581,mean_tpm_UBERON_0000966,mean_tpm_UBERON_0002367,mean_tpm_UBERON_0001987,mean_tpm_UBERON_0000992,mean_tpm_UBERON_0001043,mean_tpm_UBERON_0000002,mean_tpm_UBERON_0001255
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""FCHS_1""",0.900785,0.401732,0.001687,0.847295,0.005244,0.019244,0.003343,1.019749,0.0,0.0,0.0,0.000852,0.205011,0.0,0.0,0.012766,0.0,0.077015,0.518301,0.0,0.386322,0.0,0.490501,0.0,0.0,0.007051,0.0,0.203287,0.0,0.081244,0.0,0.0,0.071592,0.0,0.0,0.216013,…,0.0,0.0,0.0,0.220624,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.210284,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.088743,0.0,0.0,0.131108,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_3""",0.048998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010173,0.0,0.017463,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


&nbsp;

&nbsp;

### 7. test で実行した THP-1 の計算との一致を確認する

- 最初に、EFO 名で選んだ元の試料群と、Cellosaurus ID で選んだ群が同じ試料集合か確認する
- 違っていれば、計算ライブラリーではなく**対象試料の違い**なので停止する
- 一致すれば全 CRE の THP-1 平均を元の DataFrame と比較する。入力値や集約順序を変更していないため、ここは完全一致で調査

In [20]:
from polars.testing import assert_frame_equal

thp1_mean_column = fantabio_tpm_thp1.columns[1]

thp1_sources_all = (
    fantabio_group_metadata.filter(pl.col("mean_column") == thp1_mean_column)
    .select("source_names").explode("source_names")["source_names"].to_list()
)
assert sorted(thp1_sources_all) == sorted(THP1_cell_source_names), "THP-1 の対象試料が違います。"

fantabio_thp1_check = fantabio_tpm_mean.select("cre_id", thp1_mean_column).collect(engine="in-memory")


# Horizontal means can differ by about 1e-16 when the Polars plan differs.
assert_frame_equal(
    fantabio_thp1_check,
    fantabio_tpm_thp1,
    abs_tol=1e-8,
    check_exact=False,
)


display(fantabio_thp1_check.head(5))
print("THP-1: 対象試料と全 CRE の平均 TPM が abs_tol=1e-8 で一致しました。")

cre_id,mean_tpm_CVCL_0006
str,f64
"""FCHS_1""",0.847295
…,…
"""FCHS_3""",0.0


THP-1: 対象試料と全 CRE の平均 TPM が abs_tol=1e-8 で一致しました。


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 8. 特異性の比較対象を選ぶ

- 既定値は、上で作成した **全種類の annotation 付き biosample 群
- この混合集合での値は、あくまでサンプルレベルでまとめたものであり、純粋な組織特異性 ではないことに注意
- 組織だけの解析にする場合は下のコメント行に置き換える
- その場合、群数と比較対象が変わるためスコアも変わる

In [21]:
specificity_groups = fantabio_group_metadata
# 組織のみを比較する場合:
# specificity_groups = fantabio_group_metadata.filter(
#     pl.col("group_type") == "tissue"
# )

mean_tpm_columns = specificity_groups["mean_column"].to_list()

assert len(mean_tpm_columns) >= 2, "特異性の計算には2群以上が必要です。"

display(
    specificity_groups.select(
        "group_id", "group_names", "group_type", "n_samples"
    )
)

group_id,group_names,group_type,n_samples
str,list[str],str,u32
"""CVCL_0031""","[""MCF-7""]","""cell_line""",256
…,…,…,…
"""UBERON_0001255""","[""urinary bladder""]","""tissue""",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 9. 各 CRE の group 別平均を、一つの List 列にまとめる

- `group_tpm` は Python のリストへ変換したものではなく、**Polars の List 型の列**
- リストの要素の順番は `specificity_groups` の行順と対応する
- List にすることで、元の巨大な行列を `unpivot` せず、行ごとに entropy を計算する

In [22]:
fantabio_group_values = fantabio_tpm_mean.select(
    "cre_id", pl.concat_list(mean_tpm_columns).alias("group_tpm"),
)

display(fantabio_group_values.head(5).collect())

cre_id,group_tpm
str,list[f64]
"""FCHS_1""","[0.900785, 0.401732, … 0.0]"
…,…
"""FCHS_3""","[0.048998, 0.0, … 0.0]"


In [23]:
fantabio_group_values = fantabio_group_values.with_columns(
    pl.col("group_tpm").list.len().alias("n_groups"),
    pl.col("group_tpm").list.sum().alias("sum_group_tpm"),
    pl.col("group_tpm").list.max().alias("max_group_tpm"),
)

display(fantabio_group_values.head(5).collect())

cre_id,group_tpm,n_groups,sum_group_tpm,max_group_tpm
str,list[f64],u32,f64,f64
"""FCHS_1""","[0.900785, 0.401732, … 0.0]",519,12.221825,1.092198
…,…,…,…,…
"""FCHS_3""","[0.048998, 0.0, … 0.0]",519,0.481988,0.386147


&nbsp;

&nbsp;

&nbsp;

### 処理の追加

- 実装の参考にしている [04expressionSpecificity.R](https://github.com/eddieimada/fcr2/blob/master/GTEX/04expressionSpecificity.R) の `entropyFunc` では、グループでの平均が 0.1 TPM 未満なら 0 として扱う処理があるので、これを付け加える
- Rscript では、[entropy.empirical](https://www.rdocumentation.org/packages/entropy/versions/1.3.2/topics/entropy.empirical) が使用されている

In [24]:
fantabio_group_values = (
    fantabio_group_values
    .with_columns(
        pl.col("group_tpm")
        .list.eval(
            pl.when(pl.element() < 0.1)
            .then(0.0)
            .otherwise(pl.element())
        )
        .alias("group_tpm")
    )
    .with_columns(
        pl.col("group_tpm").list.sum().alias("sum_group_tpm")
    )
)

display(fantabio_group_values.head(5).collect())

cre_id,group_tpm,n_groups,sum_group_tpm,max_group_tpm
str,list[f64],u32,f64,f64
"""FCHS_1""","[0.900785, 0.401732, … 0.0]",519,11.165756,1.092198
…,…,…,…,…
"""FCHS_3""","[0.0, 0.0, … 0.0]",519,0.386147,0.386147


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 10. Polars の `Expr.entropy` を使う

- `list.eval` の内側は CRE **一行分**の群別 TPM
- `normalize=True` により群間の比率へ正規化する
- 自然対数の entropy を求める
- `value_counts` やヒストグラムから計算するのではないことに注意！
- 0 は $0\log0=0$ なので entropy の内部では正の要素だけを渡す
- **`n_groups` はゼロの群を含む元の長さのまま** にしておく
- 群平均 TPM が 0.1 未満の値は、entropy 計算前にゼロへ置換する
- 閾値処理後に全群ゼロとなる CRE は、entropy を `null` にする
- ここで使うのは empirical Shannon entropy (底2の empirical Shannon entropy を求める（`base=2.0`）) であり、Chao–Shen 補正ではないことに注意

In [25]:
fantabio_entropy = (
    fantabio_group_values
    .with_columns(
        pl
        .when(pl.col("sum_group_tpm") > 0)
        .then(
        pl.col("group_tpm")
                    .list
                    .eval(pl.element().filter(pl.element() > 0) # ゼロは entropy に含めない
                    .entropy(
                        base=2,
                        normalize=True
                    )).list.first())
        
        .otherwise(None).alias("entropy"),
)
)

display(fantabio_entropy.head(5).collect())

cre_id,group_tpm,n_groups,sum_group_tpm,max_group_tpm,entropy
str,list[f64],u32,f64,f64,f64
"""FCHS_1""","[0.900785, 0.401732, … 0.0]",519,11.165756,1.092198,4.259368
…,…,…,…,…,…
"""FCHS_3""","[0.0, 0.0, … 0.0]",519,0.386147,0.386147,0.0


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 11. Expression specificity を計算する

$$S=1-\frac{H}{\ln K}$$

- 群平均 TPM を計算した後、0.1 未満の群平均をゼロへ置換する
- H は、閾値処理後の群平均に基づく empirical Shannon entropy
- K は `n_groups`。閾値処理でゼロになった群も含める
- 閾値処理後、全群で同じ正の値なら S=0、一群のみ正なら S=1
- 閾値処理後に全群ゼロとなる CRE は `null` とする
- pseudo-count の追加や、入力 TPM の log2 変換は行わない
- `clip` は丸め誤差による僅かな範囲外の値を防ぐ

In [26]:
fantabio_specificity = (
    fantabio_entropy
    .with_columns(
        (
            1.0
            - (
                pl.col("entropy")
                / (
                    pl.col("n_groups")
                    .cast(pl.Float64)
                    .log(base=2.0)
                )
            )
        )
        .clip(0.0, 1.0)
        .alias("specificity")
    )
    .drop("group_tpm")
).sort("specificity", descending=True)

display(fantabio_specificity.head(10).collect())

cre_id,n_groups,sum_group_tpm,max_group_tpm,entropy,specificity
str,u32,f64,f64,f64,f64
"""FCHS_290690""",519,0.0,0.048227,null,null
…,…,…,…,…,…
"""FCHS_448642""",519,0.0,0.012048,null,null


&nbsp;

&nbsp;

&nbsp;

### 12. Join CRE  annotation 

- 結果の値を変える操作ではなく、`cre_id` による注釈の追加
- `validate="1:1"` によって重複キーによる行の増加を防ぐ

In [27]:
fantabio_specificity_bed = (
    fantabio_specificity
    .join(
    fantabio_bed_select.lazy(),
    on="cre_id",
    how="left",
    validate="1:1"
    )
).sort("specificity", descending=True)

fantabio_specificity_pla = fantabio_specificity_bed.filter(pl.col("fantabio_cre_class") == "PLA")
fantabio_specificity_ela = fantabio_specificity_bed.filter(pl.col("fantabio_cre_class") == "ELA")

display(fantabio_specificity_bed.head(5).collect())

cre_id,n_groups,sum_group_tpm,max_group_tpm,entropy,specificity,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,u32,f64,f64,f64,f64,str,f64,str,str,str
"""FCHS_290690""",519,0.0,0.048227,null,null,"""FCHS_290690""",-0.857143,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_448112""",519,0.0,0.050555,null,null,"""FCHS_448112""",1.0,"""ELA""","""255,202,74""","""yellow"""


In [28]:
fantabio_non_specificity_bed= fantabio_specificity_bed.filter(pl.col("specificity").is_null())

display(fantabio_non_specificity_bed.collect())

cre_id,n_groups,sum_group_tpm,max_group_tpm,entropy,specificity,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,u32,f64,f64,f64,f64,str,f64,str,str,str
"""FCHS_447314""",519,0.0,0.034096,null,null,"""FCHS_447314""",-1.0,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_500""",519,0.0,0.032055,null,null,"""FCHS_500""",0.2,"""ELA""","""255,202,74""","""yellow"""


In [29]:
fantabio_specificity_bed_filtered = fantabio_specificity_bed.filter(
    pl.col("specificity").is_not_null()
)

display(fantabio_specificity_bed_filtered.collect())

cre_id,n_groups,sum_group_tpm,max_group_tpm,entropy,specificity,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,u32,f64,f64,f64,f64,str,f64,str,str,str
"""FCHS_3""",519,0.386147,0.386147,0.0,1.0,"""cp2@LOC101928626""",-1.0,"""ELA""","""255,202,74""","""yellow"""
…,…,…,…,…,…,…,…,…,…,…
"""FCHS_10496""",519,26032.850408,133.935532,8.95079,0.007628,"""cp1@TRAPPC3""",-0.999564,"""PLA""","""0,0,204""","""blue"""


In [30]:
# check
FCHS_354464 = fantabio_specificity_bed_filtered.filter(pl.col("cre_id") == "FCHS_354464")
display(FCHS_354464.collect())

cre_id,n_groups,sum_group_tpm,max_group_tpm,entropy,specificity,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,u32,f64,f64,f64,f64,str,f64,str,str,str
"""FCHS_354464""",519,145627.935737,19227.190798,7.166478,0.205454,"""cp2@HSPA1A,cp2@HSPA1L""",0.95538,"""PLA""","""204,0,0""","""red"""
